In [51]:
import cx_Oracle
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

In [47]:
%matplotlib inline
sns.set_style('whitegrid')
from matplotlib import font_manager
# font_path='/usr/share/fonts/cjkuni-uming/uming.ttc'
font_path = '/usr/share/fonts/truetype/arphic/uming.ttc'
matplotlib.rcParams['font.family']=font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus']=False

In [48]:
import warnings
warnings.filterwarnings('ignore')

In [49]:
%matplotlib inline

In [53]:
import os
print("LD_LIBRARY_PATH:", os.getenv('LD_LIBRARY_PATH'))
print("ORACLE_HOME:", os.getenv('ORACLE_HOME'))

LD_LIBRARY_PATH: /oracle/client:/home/quant/oracle ORACLE_HOME=/oracle/client
ORACLE_HOME: None


### 批量处理所有股票

In [40]:
# 获取利润表中的季度报或者年报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHAREINCOME where  STATEMENT_TYPE in (408001000,408005000,408027000,408028000,408036000,408045000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

# 获取利润表中的包含更正和调整数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHAREINCOME where STATEMENT_TYPE in (408001000,408004000,408050000,408029000,408031000,408037000,408046000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df3 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [52]:
# 获取利润表中的单季度报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHAREINCOME where STATEMENT_TYPE ='408002000'"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

df3=df1

In [36]:
start_date=df1['REPORT_PERIOD'].min()
end_date='20250626'
print(f"开始日期：{start_date},结束日期：{end_date}")

开始日期：20090331,结束日期：20250626


In [20]:
#生成日期序列
date_range=pd.date_range(start=start_date,end=end_date,freq='D')
df_dates=pd.DataFrame({'date':date_range})
#格式化日期
df_dates['Date']=df_dates['date'].dt.strftime("%Y%m%d")
df_dates['Date']=pd.to_datetime(df_dates['Date'])
df_dates=df_dates['Date']

In [21]:
df_dates

0      2009-03-31
1      2009-04-01
2      2009-04-02
3      2009-04-03
4      2009-04-04
          ...    
5927   2025-06-22
5928   2025-06-23
5929   2025-06-24
5930   2025-06-25
5931   2025-06-26
Name: Date, Length: 5932, dtype: datetime64[ns]

In [23]:
print(f"df1:{df1.shape},df3:{df3.shape}")

df1:(308923, 114),df3:(543187, 114)


In [24]:
df3['STATEMENT_TYPE'].unique()

array([408001000, 408004000, 408050000, 408029000, 408031000])

In [25]:
def prepare_data(df_merged,df4,period_offset):
    """
    准备原始季度数据：df_quarterly
    准备包含原始和调整数据：df_yoy
    period_offset:报告期偏移量
    """
    # 2. 计算实际报告间隔（动态替代固定3个月）
    df_merged['REPORT_PERIOD']=pd.to_datetime(df_merged['REPORT_PERIOD'])
    df_quarterly = df_merged.drop_duplicates(subset='REPORT_PERIOD').copy()
    df_quarterly['next_REPORT_PERIOD'] = df_quarterly['REPORT_PERIOD'].shift(period_offset)  # 获取下一次报告的实际日期
    df_quarterly['actual_interval'] = (df_quarterly['next_REPORT_PERIOD'] - df_quarterly['REPORT_PERIOD']).dt.days
    
    df_yoy=df4.copy()
    df_yoy['IS_ADJUSTED'] = df_yoy['STATEMENT_TYPE'].isin(['408004000', '408050000','408029000','408031000'])
    df_yoy['REPORT_PERIOD']=pd.to_datetime(df_yoy['REPORT_PERIOD'])
    df_yoy['ACTUAL_ANN_DT']=pd.to_datetime(df_yoy['ACTUAL_ANN_DT'])
    
    return df_quarterly,df_yoy

In [26]:
def create_next_period_dict(df_quarterly):
    """
    创建报告期与下一个报告期对应的公告日期的映射字典
    """
    # 创建一个字典，用于存储每个报告期对应的下一个报告期的ACTUAL_ANN_DT
    next_period_ann_dt = {}
    for idx, row in df_quarterly.iterrows():
        if pd.notna(row['next_REPORT_PERIOD']):
            # 查找下一个报告期对应的行
            next_period_rows = df_quarterly[df_quarterly['REPORT_PERIOD'] == row['next_REPORT_PERIOD']]
            if not next_period_rows.empty:
                next_period_ann_dt[row['REPORT_PERIOD']] = next_period_rows.iloc[0]['ACTUAL_ANN_DT']
    return next_period_ann_dt

In [27]:
def apply_adjusted_data(df_quarterly,df_yoy,next_period_ann_dt):
    """
    判断是否应用调整数据到季报数据中，并返回结果
    """
    result_df = df_quarterly.copy()
    
     # 遍历df_quarterly中的每一行
    for idx, orig_row in df_quarterly.iterrows():
        report_period = orig_row['REPORT_PERIOD']
        # 查找df_yoy中对应报告期的所有行（可能包含原始数据和多次调整数据）
        adj_rows = df_yoy[(df_yoy['REPORT_PERIOD'] == report_period) & (df_yoy['IS_ADJUSTED'] == True)]
        if not adj_rows.empty:
            # 获取下一个报告期的公告日期
            next_ann_dt = next_period_ann_dt.get(report_period)
            if next_ann_dt is not None:
                # 筛选出公告日期大于下一个报告期公告日期的调整数据
                valid_adj_rows = adj_rows[adj_rows['ACTUAL_ANN_DT'] <= next_ann_dt]
                if not valid_adj_rows.empty:
                    # 使用最新的有效调整数据
                    latest_adj_row = valid_adj_rows.iloc[-1]
                    # 更新结果DataFrame中的财务数据列
                    financial_columns = [col for col in result_df.columns]
                    
                    for col in financial_columns:
                        if col in latest_adj_row:
                            result_df.at[idx, col] = latest_adj_row[col]
    return result_df
    

In [28]:
def create_shifted_data(df_merged,result_df):
    """
    创建平移后的数据
    """
    # 3. 构建动态偏移映射表
    period_map = result_df.set_index('REPORT_PERIOD')['next_REPORT_PERIOD'].to_dict()
    
    # 4. 标记需要平移的列
    non_financial_columns = ['Date', 'REPORT_PERIOD']
    financial_columns = [col for col in df_merged.columns if col not in non_financial_columns]
    
    # 5. 创建平移后的DataFrame
    df_shifted = df_merged.copy()
    
    for col in financial_columns:
        # 为每个财务值找到下一次报告期的值
        df_quarterly_shifted = result_df.copy()
        df_quarterly_shifted['mapped_REPORT_PERIOD'] = df_quarterly_shifted['REPORT_PERIOD'].map(period_map)
        df_quarterly_shifted['mapped_REPORT_PERIOD'] = pd.to_datetime(df_quarterly_shifted['mapped_REPORT_PERIOD'])
        
        # 动态匹配
        temp_df = pd.merge_asof(
            df_merged[['REPORT_PERIOD']].sort_values('REPORT_PERIOD'),
            df_quarterly_shifted[['mapped_REPORT_PERIOD', col]]
                .dropna()
                .sort_values('mapped_REPORT_PERIOD'),
            left_on='REPORT_PERIOD',
            right_on='mapped_REPORT_PERIOD',
            direction='backward'
        )
        df_shifted[col] = temp_df[col].values
        
    df_shifted=df_shifted.dropna(subset=['S_INFO_WINDCODE'])

    #构建逆向字典
    reverse_period_map={v:k for k,v in period_map.items() if pd.notnull(v)}
    
    #映射回原始值
    df_shifted['REPORT_PERIOD']=df_shifted['REPORT_PERIOD'].map(reverse_period_map)
    
    return df_shifted

In [29]:
#获取所有股票代码
# stock_ids=df1['S_INFO_WINDCODE'].unique()
stock_ids=['600000.SH','600100.SH']
results=[]
results_shift=[]

In [30]:
for stock_id in tqdm(stock_ids,desc='Processing stocks'):
    #提取当前股票的年报数据并按实际公布日期排序
    stock_annual=df1[df1['S_INFO_WINDCODE']==stock_id].sort_values('ACTUAL_ANN_DT')
    
    stock_annual['ACTUAL_ANN_DT']=pd.to_datetime(stock_annual['ACTUAL_ANN_DT'])
    
    #合并两个数据框
    df_merged=pd.merge_asof(df_dates,stock_annual,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')
    df_merged=df_merged.dropna(subset=['S_INFO_WINDCODE'])
    
    df4=df3[df3['S_INFO_WINDCODE']==stock_id]
    
    df_quarterly,df_yoy=prepare_data(df_merged, df4,period_offset=-2)
    
    # 创建一个字典，用于存储每个报告期对应的下一个报告期的ACTUAL_ANN_DT
    next_period_ann_dt = create_next_period_dict(df_quarterly)
                
    # 构造偏移时间轴数据时，调整数据应用
    result_df = apply_adjusted_data(df_quarterly,df_yoy,next_period_ann_dt)
    
    #创建平移后数据
    df_shifted=create_shifted_data(df_merged,result_df)

    results.append(df_merged)
    
    results_shift.append(df_shifted)
    
    # del stock_annual,df_quarterly,df_yoy,next_period_ann_dt,result_df
    
    

Processing stocks: 100%|██████████| 2/2 [00:01<00:00,  1.68it/s]


In [32]:
df4[['S_INFO_WINDCODE','ANN_DT','REPORT_PERIOD','STATEMENT_TYPE','ACTUAL_ANN_DT']]

,S_INFO_WINDCODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,ACTUAL_ANN_DT
1108,600100.SH,20090428,20090331,408001000,20090428
3088,600100.SH,20100428,20090331,408004000,20100428
4327,600100.SH,20090812,20090630,408001000,20090812
7629,600100.SH,20100828,20090630,408004000,20100828
9040,600100.SH,20091029,20090930,408001000,20091029
...,...,...,...,...,...
516120,600100.SH,20250429,20240331,408004000,20250429
524247,600100.SH,20240831,20240630,408001000,20240831
528629,600100.SH,20241030,20240930,408001000,20241030
536286,600100.SH,20250429,20241231,408001000,20250429


In [33]:
df_merged

,Date,OBJECT_ID,S_INFO_WINDCODE,WIND_CODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,TOT_OPER_REV,OPER_REV,...,IS_CALCULATION,OTHER_IMPAIR_LOSS_ASSETS,TOT_OPER_COST2,AMODCOST_FIN_ASSETS,TOT_OPT_INC_DIF,TOT_OPT_INC_DIF_MEMO,TOT_OPT_COST_DIF,TOT_OPT_COST_DIF_MEMO,OPDATE,OPMODE
28,2009-04-28,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,None,NaN,None,2019-09-24 20:22:09,0
29,2009-04-29,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,None,NaN,None,2019-09-24 20:22:09,0
30,2009-04-30,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,None,NaN,None,2019-09-24 20:22:09,0
31,2009-05-01,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,None,NaN,None,2019-09-24 20:22:09,0
32,2009-05-02,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,None,NaN,None,2019-09-24 20:22:09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5927,2025-06-22,{52A26BCF-2438-11F0-9A1C-96A468826B84},600100.SH,600100.SH,20250429,2025-03-31,408001000.0,CNY,2.019044e+09,2.019044e+09,...,0.0,NaN,2.275901e+09,NaN,NaN,None,NaN,None,2025-04-30 19:31:27,1
5928,2025-06-23,{52A26BCF-2438-11F0-9A1C-96A468826B84},600100.SH,600100.SH,20250429,2025-03-31,408001000.0,CNY,2.019044e+09,2.019044e+09,...,0.0,NaN,2.275901e+09,NaN,NaN,None,NaN,None,2025-04-30 19:31:27,1
5929,2025-06-24,{52A26BCF-2438-11F0-9A1C-96A468826B84},600100.SH,600100.SH,20250429,2025-03-31,408001000.0,CNY,2.019044e+09,2.019044e+09,...,0.0,NaN,2.275901e+09,NaN,NaN,None,NaN,None,2025-04-30 19:31:27,1
5930,2025-06-25,{52A26BCF-2438-11F0-9A1C-96A468826B84},600100.SH,600100.SH,20250429,2025-03-31,408001000.0,CNY,2.019044e+09,2.019044e+09,...,0.0,NaN,2.275901e+09,NaN,NaN,None,NaN,None,2025-04-30 19:31:27,1


In [34]:
results_shift[1]

,Date,OBJECT_ID,S_INFO_WINDCODE,WIND_CODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,TOT_OPER_REV,OPER_REV,...,IS_CALCULATION,OTHER_IMPAIR_LOSS_ASSETS,TOT_OPER_COST2,AMODCOST_FIN_ASSETS,TOT_OPT_INC_DIF,TOT_OPT_INC_DIF_MEMO,TOT_OPT_COST_DIF,TOT_OPT_COST_DIF_MEMO,OPDATE,OPMODE
212,2009-10-29,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,NaN,NaN,NaN,2019-09-24 20:22:09,0
213,2009-10-30,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,NaN,NaN,NaN,2019-09-24 20:22:09,0
214,2009-10-31,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,NaN,NaN,NaN,2019-09-24 20:22:09,0
215,2009-11-01,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,NaN,NaN,NaN,2019-09-24 20:22:09,0
216,2009-11-02,{DF628415-D92B-4375-B187-90F5745BFE87},600100.SH,600100.SH,20090428,2009-03-31,408001000.0,CNY,2.298999e+09,2.298999e+09,...,0.0,NaN,2.385244e+09,NaN,NaN,NaN,NaN,NaN,2019-09-24 20:22:09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5927,2025-06-22,{015A0638-66D1-11EF-868C-B8CA3A6626A7},600100.SH,600100.SH,20240831,2024-06-30,408001000.0,CNY,6.303196e+09,6.303196e+09,...,0.0,NaN,6.710530e+09,0.0,NaN,NaN,NaN,NaN,2024-08-30 21:09:04,0
5928,2025-06-23,{015A0638-66D1-11EF-868C-B8CA3A6626A7},600100.SH,600100.SH,20240831,2024-06-30,408001000.0,CNY,6.303196e+09,6.303196e+09,...,0.0,NaN,6.710530e+09,0.0,NaN,NaN,NaN,NaN,2024-08-30 21:09:04,0
5929,2025-06-24,{015A0638-66D1-11EF-868C-B8CA3A6626A7},600100.SH,600100.SH,20240831,2024-06-30,408001000.0,CNY,6.303196e+09,6.303196e+09,...,0.0,NaN,6.710530e+09,0.0,NaN,NaN,NaN,NaN,2024-08-30 21:09:04,0
5930,2025-06-25,{015A0638-66D1-11EF-868C-B8CA3A6626A7},600100.SH,600100.SH,20240831,2024-06-30,408001000.0,CNY,6.303196e+09,6.303196e+09,...,0.0,NaN,6.710530e+09,0.0,NaN,NaN,NaN,NaN,2024-08-30 21:09:04,0


In [44]:

WIND_CONFIG = {
    'username': "wind",
    'password': "wind",
    'host': "10.6.60.114",
    'port': "1521",
    'service_name': "wind"
}

In [45]:
import cx_Oracle
def get_table_comments(target_config, table_name):
    try:
        # 1. 连接数据库
        conn = cx_Oracle.connect(
            f"{target_config['username']}/{target_config['password']}@{target_config['host']}:{target_config['port']}/{target_config['service_name']}")
        cursor = conn.cursor()
        
        # 2. 构建SQL查询
        sql = """
        SELECT cols.column_name, comm.comments
        FROM user_tab_columns cols
        LEFT JOIN user_col_comments comm
            ON cols.table_name = comm.table_name 
            AND cols.column_name = comm.column_name
        WHERE cols.table_name = UPPER(:tb_name)
        ORDER BY cols.column_id
        """
        
        # 3. 执行查询并返回字典
        cursor.execute(sql, tb_name=table_name)
        field_map = {row[0]: row[1] for row in cursor}
        return field_map
        
    except cx_Oracle.Error as e:
        print(f"数据库错误: {e}")
    finally:
        if 'cursor' in locals():
            cursor.close()
        if 'conn' in locals():
            conn.close()

# 使用示例
if __name__ == "__main__":
    table_comments = get_table_comments(WIND_CONFIG, 'ASHAREINCOME')
    print(table_comments)

{'OBJECT_ID': '对象ID', 'S_INFO_WINDCODE': 'Wind代码', 'WIND_CODE': 'Wind代码', 'ANN_DT': '公告日期', 'REPORT_PERIOD': '报告期', 'STATEMENT_TYPE': '报表类型', 'CRNCY_CODE': '货币代码', 'TOT_OPER_REV': '营业总收入', 'OPER_REV': '营业收入', 'INT_INC': '利息收入', 'NET_INT_INC': '利息净收入', 'INSUR_PREM_UNEARNED': '已赚保费', 'HANDLING_CHRG_COMM_INC': '手续费及佣金收入', 'NET_HANDLING_CHRG_COMM_INC': '手续费及佣金净收入', 'NET_INC_OTHER_OPS': '其他经营净收益', 'PLUS_NET_INC_OTHER_BUS': '加:其他业务净收益', 'PREM_INC': '保费业务收入', 'LESS_CEDED_OUT_PREM': '减:分出保费', 'CHG_UNEARNED_PREM_RES': '提取未到期责任准备金', 'INCL_REINSURANCE_PREM_INC': '其中:分保费收入', 'NET_INC_SEC_TRADING_BROK_BUS': '代理买卖证券业务净收入', 'NET_INC_SEC_UW_BUS': '证券承销业务净收入', 'NET_INC_EC_ASSET_MGMT_BUS': '受托客户资产管理业务净收入', 'OTHER_BUS_INC': '其他业务收入', 'PLUS_NET_GAIN_CHG_FV': '加:公允价值变动净收益', 'PLUS_NET_INVEST_INC': '加:投资净收益', 'INCL_INC_INVEST_ASSOC_JV_ENTP': '其中:对联营企业和合营企业的投资收益', 'PLUS_NET_GAIN_FX_TRANS': '加:汇兑净收益', 'TOT_OPER_COST': '营业总成本', 'LESS_OPER_COST': '减:营业成本', 'LESS_INT_EXP': '减:利息支出', 'LESS_HANDLING_CHRG_COMM_EXP':